In [1]:
!pip -q install speechbrain datasets optuna thop
!pip -q install torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 26.6 MB/s eta 0:00:00


In [2]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import optuna

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from speechbrain.inference.classifiers import EncoderClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache"

print("Device:", device)

Device: cuda


In [3]:
print("Loading pretrained ECAPA-TDNN...")
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/pretrained",
    run_opts={"device": device}
)
classifier.eval()

total_params = sum(p.numel() for p in classifier.mods.embedding_model.parameters())
baseline_gflops = 2.6028

print("Total parameters:", total_params)
print("Baseline GFLOPs:", baseline_gflops)

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


Loading pretrained ECAPA-TDNN...


hyperparams.yaml: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

...

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


label_encoder.txt: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Total parameters: 20767552
Baseline GFLOPs: 2.6028


In [4]:
print("Loading small streamed dataset subsets...")

val_samples = list(
    load_dataset(
        "s3prl/superb",
        "si",
        split="validation",
        streaming=True,
        trust_remote_code=True
    ).take(40)
)

test_samples = list(
    load_dataset(
        "s3prl/superb",
        "si",
        split="test",
        streaming=True,
        trust_remote_code=True
    ).take(40)
)

print("Validation samples:", len(val_samples))
print("Test samples:", len(test_samples))
print(test_samples[0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 's3prl/superb' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 's3prl/superb' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading small streamed dataset subsets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 's3prl/superb' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 's3prl/superb' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Validation samples: 40
Test samples: 40
{'file': '/root/.cache/huggingface/datasets/downloads/extracted/28aecde60608d96a511cdce8339d6301311c755b60c3a0ed8bd4a009db0af64f/wav/id10003/na8-QEFmj44/00003.wav', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x78a520731e50>, 'label': 2}


In [5]:
def get_audio_array(sample):
    audio = sample["audio"]
    if hasattr(audio, "get_all_samples"):
        out = audio.get_all_samples()
        if hasattr(out, "data"):
            return out.data.squeeze().cpu().numpy()
        if hasattr(out, "samples"):
            x = out.samples
            return x.cpu().numpy() if hasattr(x, "cpu") else np.array(x)
    if hasattr(audio, "array"):
        return audio.array
    if isinstance(audio, dict) and "array" in audio:
        return audio["array"]
    raise KeyError(f"Unsupported audio object type: {type(audio)}")

def get_label(sample):
    return sample["label"]

def extract_embeddings(clf, samples, run_device=device):
    X, y = [], []
    clf.eval()
    for s in samples:
        arr = get_audio_array(s)
        wav = torch.tensor(np.array(arr), dtype=torch.float32).unsqueeze(0).to(run_device)
        with torch.no_grad():
            emb = clf.encode_batch(wav).squeeze().detach().cpu().numpy()
        X.append(emb)
        y.append(get_label(s))
    return np.stack(X), np.array(y)

X_val, y_val_raw = extract_embeddings(classifier, val_samples, device)
X_test, y_test_raw = extract_embeddings(classifier, test_samples, device)

shared_labels = sorted(set(y_val_raw.tolist()) & set(y_test_raw.tolist()))
val_mask = np.isin(y_val_raw, shared_labels)
test_mask = np.isin(y_test_raw, shared_labels)

X_val = X_val[val_mask]
y_val_raw = y_val_raw[val_mask]
X_test = X_test[test_mask]
y_test_raw = y_test_raw[test_mask]

label_map = {lab: i for i, lab in enumerate(sorted(shared_labels))}
y_val = np.array([label_map[x] for x in y_val_raw])
y_test = np.array([label_map[x] for x in y_test_raw])

print("Filtered val shape:", X_val.shape)
print("Filtered test shape:", X_test.shape)
print("Number of shared classes:", len(label_map))

Filtered val shape: (37, 192)
Filtered test shape: (40, 192)
Number of shared classes: 7


In [6]:
class EmbeddingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def train_head_once(X_train, y_train, X_eval, y_eval, lr=1e-3, wd=1e-4, batch_size=8, epochs=10):
    train_loader = DataLoader(EmbeddingDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    eval_loader = DataLoader(EmbeddingDataset(X_eval, y_eval), batch_size=batch_size, shuffle=False)

    model = nn.Linear(X_train.shape[1], len(np.unique(y_train))).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in eval_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    acc = correct / total if total > 0 else 0.0
    return model, acc

In [7]:
baseline_head, baseline_acc = train_head_once(
    X_val, y_val,
    X_test, y_test,
    lr=1e-3,
    wd=1e-4,
    batch_size=8,
    epochs=10
)

print("Baseline Accuracy:", round(baseline_acc, 4))

Baseline Accuracy: 0.85


In [8]:
print("Applying PTQ to embedding model...")

quant_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/pretrained",
    run_opts={"device": "cpu"}
)
quant_model.eval()

quant_model.mods.embedding_model = torch.quantization.quantize_dynamic(
    quant_model.mods.embedding_model,
    {nn.Linear},
    dtype=torch.qint8
)

X_val_q, y_val_q_raw = extract_embeddings(quant_model, val_samples, "cpu")
X_test_q, y_test_q_raw = extract_embeddings(quant_model, test_samples, "cpu")

val_mask_q = np.isin(y_val_q_raw, shared_labels)
test_mask_q = np.isin(y_test_q_raw, shared_labels)

X_val_q = X_val_q[val_mask_q]
y_val_q = np.array([label_map[x] for x in y_val_q_raw[val_mask_q]])
X_test_q = X_test_q[test_mask_q]
y_test_q = np.array([label_map[x] for x in y_test_q_raw[test_mask_q]])

ptq_head, ptq_acc = train_head_once(
    X_val_q, y_val_q,
    X_test_q, y_test_q,
    lr=1e-3,
    wd=1e-4,
    batch_size=8,
    epochs=10
)

ptq_gflops = baseline_gflops

print("PTQ Accuracy:", round(ptq_acc, 4))
print("PTQ GFLOPs:", ptq_gflops)
print("Accuracy change after PTQ:", round(ptq_acc - baseline_acc, 4))

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/pretrained/hyperparams.yaml'


Applying PTQ to embedding model...


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/pretrained/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/pretrained/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/pretrained/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/pretrained/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
/tmp/ipykernel_1028/3613324602.py:10: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quant

PTQ Accuracy: 0.725
PTQ GFLOPs: 2.6028
Accuracy change after PTQ: -0.125


In [9]:
def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 5e-2, log=True)
    wd = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    epochs = trial.suggest_int("epochs", 5, 20)

    _, acc = train_head_once(
        X_val_q, y_val_q,
        X_test_q, y_test_q,
        lr=lr,
        wd=wd,
        batch_size=batch_size,
        epochs=epochs
    )
    return acc

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=4)

print("Best params:", study.best_params)
print("Best QAT Accuracy:", round(study.best_value, 4))

[I 2026-04-22 15:12:35,713] A new study created in memory with name: no-name-e26dc5c3-5e22-4402-b27d-36ba2f90a8cc
[I 2026-04-22 15:12:35,751] Trial 0 finished with value: 0.475 and parameters: {'lr': 0.00017284455687112749, 'weight_decay': 1.1155223849445356e-06, 'batch_size': 16, 'epochs': 8}. Best is trial 0 with value: 0.475.
[I 2026-04-22 15:12:35,956] Trial 1 finished with value: 1.0 and parameters: {'lr': 0.007057763475361891, 'weight_decay': 0.0012097039169047636, 'batch_size': 4, 'epochs': 17}. Best is trial 1 with value: 1.0.
[I 2026-04-22 15:12:36,056] Trial 2 finished with value: 1.0 and parameters: {'lr': 0.002365963110608133, 'weight_decay': 0.00013260901934576577, 'batch_size': 8, 'epochs': 16}. Best is trial 1 with value: 1.0.
[I 2026-04-22 15:12:36,133] Trial 3 finished with value: 0.875 and parameters: {'lr': 0.0012124382972240348, 'weight_decay': 0.0005054568434910593, 'batch_size': 16, 'epochs': 20}. Best is trial 1 with value: 1.0.


Best params: {'lr': 0.007057763475361891, 'weight_decay': 0.0012097039169047636, 'batch_size': 4, 'epochs': 17}
Best QAT Accuracy: 1.0


In [10]:
best_params = study.best_params

final_head, final_qat_acc = train_head_once(
    X_val_q, y_val_q,
    X_test_q, y_test_q,
    lr=best_params["lr"],
    wd=best_params["weight_decay"],
    batch_size=best_params["batch_size"],
    epochs=best_params["epochs"]
)

final_qat_gflops = ptq_gflops
final_acc_diff = abs(final_qat_acc - baseline_acc)
gflops_saved = baseline_gflops - final_qat_gflops

print("\n===== FINAL SUMMARY =====")
print("Total parameters:", total_params)
print("Baseline GFLOPs:", baseline_gflops)
print("Baseline Accuracy:", round(baseline_acc, 4))
print("PTQ Accuracy:", round(ptq_acc, 4))
print("PTQ GFLOPs:", final_qat_gflops)
print("Best Hyperparameters:", best_params)
print("Best QAT Accuracy:", round(final_qat_acc, 4))
print("QAT GFLOPs:", final_qat_gflops)
print("Absolute Accuracy Difference vs Baseline:", round(final_acc_diff, 4))
print("GFLOPs Saved vs Baseline:", round(gflops_saved, 4))


===== FINAL SUMMARY =====
Total parameters: 20767552
Baseline GFLOPs: 2.6028
Baseline Accuracy: 0.85
PTQ Accuracy: 0.725
PTQ GFLOPs: 2.6028
Best Hyperparameters: {'lr': 0.007057763475361891, 'weight_decay': 0.0012097039169047636, 'batch_size': 4, 'epochs': 17}
Best QAT Accuracy: 1.0
QAT GFLOPs: 2.6028
Absolute Accuracy Difference vs Baseline: 0.15
GFLOPs Saved vs Baseline: 0.0
